In [ ]:
# | default_exp report

In [ ]:
# | export
from __future__ import annotations
from pathlib import Path
import structlog
import pandas as pd

log = structlog.get_logger()

In [ ]:
# | export
def generate_report(
    matrix_parquet: str | Path,
    output_dir: str | Path,
    cvd_safe: bool = False,
    shap_samples: int = 500,
    shap_features: int = 10,
) -> Path | None:
    """Generate HTML dashboard from an extracted feature matrix.

    Renders the Quarto dashboard template with interactive Plotly
    visualizations, SHAP explainability, and model validation plots.

    Requires quarto-cli (pip install quarto-cli).
    """
    matrix_parquet = Path(matrix_parquet)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not matrix_parquet.exists():
        log.error("report_matrix_not_found", path=str(matrix_parquet))
        return None

    feat_name = matrix_parquet.stem.replace("_matrix", "")

    # Import the CLI's Quarto renderer (single source of truth)
    from kreview.cli import _render_quarto_report
    import sys

    ok, msg = _render_quarto_report(
        str(matrix_parquet),
        feat_name,
        output_dir,
        sys.executable,
        cvd_safe=cvd_safe,
        shap_samples=shap_samples,
        shap_features=shap_features,
    )
    if ok:
        log.info("report_generated", output=msg)
        return Path(msg)
    else:
        log.error("report_generation_failed", error=msg)
        return None

In [ ]:
# | test
# Write simple dummy assertions
assert True